In [4]:
# Generate Urbanicity Reference Sample from GHS-SMOD
# Tyler M. Barrett | Duke Global Health Institute
# Created: June 2026
#
# Draws an equalized stratified random sample across the seven GHS-SMOD
# settlement classes (water excluded), attaches GHS-POP counts, and computes
# population-based design weights for positioning PECAHN study sites against
# the urbanicity gradient.


####################
##    PREAMBLE    ##
####################

#	Imports
import os
import arcpy

#	Check Out Spatial Analyst
if arcpy.CheckExtension("Spatial") == "Available":
	arcpy.CheckOutExtension("Spatial")
else:
	raise arcpy.ExecuteError("Spatial Analyst extension is not available.")


#######################
##  CONFIGURATION    ##
#######################

#	Project References
#	NOTE: "CURRENT" works when run from the ArcGIS Pro Python window.
#	      If running standalone via propy, replace "CURRENT" with the full
#	      path to pecahn_ref_distribution.aprx.
aprx     = arcpy.mp.ArcGISProject("CURRENT")
proj_dir = aprx.homeFolder

#	Input / Output Paths
#	NOTE: the GHS rasters are standalone GeoTIFFs in the project folder,
#	      not inside the .gdb. Epoch is E2025.
input_smod      = os.path.join(proj_dir, "GHS_SMOD_E2025_GLOBE_R2023A_54009_1000_V2_0.tif")
input_ghs_pop   = os.path.join(proj_dir, "GHS_POP_E2025_GLOBE_R2023A_54009_1000_V1_0.tif")
output_csv_path = os.path.join(proj_dir, "outputs", "ref_pts.csv")

#	Sampling Parameters
points_per_class = 200
total_classes    = 7
total_points     = points_per_class * total_classes
min_distance     = "5000 Meters"

#	Environments
#	NOTE: equal-area Mollweide + snap raster keep every grid cell-aligned;
#	      the fixed random seed makes the EQUALIZED draw reproducible.
arcpy.env.workspace              = aprx.defaultGeodatabase   # pecahn_ref_distribution.gdb
arcpy.env.overwriteOutput        = True
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(54009)   # World Mollweide
arcpy.env.snapRaster             = input_smod
arcpy.env.randomGenerator        = "42 MERSENNE_TWISTER"


############################
##  1 · RECLASSIFY SMOD   ##
############################

#	Reclassify: water (10) -> NoData; keep the seven settlement classes
print("Reclassifying SMOD raster...")
remap = arcpy.sa.RemapValue(
	[[10, "NODATA"], [11, 11], [12, 12], [13, 13],
	 [21, 21], [22, 22], [23, 23], [30, 30]]
)
smod_7_path      = os.path.join(proj_dir, "smod_7.tif")
smod_7_land_path = os.path.join(proj_dir, "smod_7_land.tif")

#	NOTE: saved as GeoTIFF in the project folder; the FGDB raster format
#	      cannot handle the ~150M-cell global 1 km grid.
smod_7 = arcpy.sa.Reclassify(input_smod, "Value", remap, "DATA")
smod_7.save(smod_7_path)

#	Mask to inhabited land using GHS-POP: any cell with a valid (non-null)
#	POP value is a GHSL land cell. This drops Antarctic ice and ocean-adjacent
#	cells that survive the water reclassify as class 11.
print("Masking to inhabited land extent...")
smod_7_land = arcpy.sa.ExtractByMask(smod_7, input_ghs_pop)
smod_7_land.save(smod_7_land_path)

#	Build attribute table and read class cell counts (N_h)
#	NOTE: N_h is taken from the masked raster so counts reflect land only.
arcpy.management.BuildRasterAttributeTable(smod_7_land_path, "Overwrite")

N_h = {}
with arcpy.da.SearchCursor(smod_7_land_path, ["Value", "Count"]) as cur:
	for value, count in cur:
		N_h[value] = count
print(f"Class cell counts (N_h): {N_h}")


############################
##  2 · DRAW SAMPLE       ##
############################

#	Equalized stratified random draw with minimum spacing to reduce
#	1 km-cell autocorrelation.
#	Parameter order: (in, out, target_field, n_points, sampling,
#	                  polygon_dimension_field, min_point_distance)
print("Generating equalized stratified random points...")
ref_pts = "ref_pts"
arcpy.sa.CreateAccuracyAssessmentPoints(
	smod_7_land_path,
	ref_pts,
	"CLASSIFIED",
	total_points,
	"EQUALIZED_STRATIFIED_RANDOM",
	None,
	min_distance
)


############################
##  3 · EXTRACT POP       ##
############################

#	Extract GHS-POP count to each point
print("Extracting population values to points...")
ref_pts_pop = "ref_pts_pop"
arcpy.sa.ExtractValuesToPoints(
	ref_pts, input_ghs_pop, ref_pts_pop, "NONE", "VALUE_ONLY"
)

#	Rename default value field
arcpy.management.AlterField(ref_pts_pop, "RASTERVALU", "pop", "Population Count")

#	Realized sample counts per class (n_h)
n_h = {k: 0 for k in N_h}
with arcpy.da.SearchCursor(ref_pts_pop, ["Classified"]) as cur:
	for (cls,) in cur:
		if cls in n_h:
			n_h[cls] += 1
print(f"Sampled counts (n_h): {n_h}")


############################
##  4 · DESIGN WEIGHTS    ##
############################

#	Per-class scale factor N_h / n_h
print("Calculating design weights...")
scale = {
	cls: (N_h[cls] / n_h[cls]) if n_h.get(cls, 0) > 0 else 0
	for cls in N_h
}

#	w = pop * (N_h / n_h); clamp NoData/negatives to zero in the weight only
arcpy.management.AddField(ref_pts_pop, "w", "DOUBLE", field_alias="Design Weight")
with arcpy.da.UpdateCursor(ref_pts_pop, ["Classified", "pop", "w"]) as cur:
	for row in cur:
		cls     = row[0]
		pop_val = row[1] if row[1] is not None else 0
		row[2]  = max(0, pop_val) * scale.get(cls, 0)
		cur.updateRow(row)


##################
##  5 · EXPORT  ##
##################

#	Add lon/lat (WGS84) so points can be located downstream.
#	Switch the SR to 54009 if your PECAHN pipeline expects Mollweide metres.
print("Adding coordinates...")
arcpy.management.CalculateGeometryAttributes(
	ref_pts_pop,
	[["lon", "POINT_X"], ["lat", "POINT_Y"]],
	coordinate_system=arcpy.SpatialReference(4326)
)

#	Export attribute table (class, pop, w, lon, lat) to CSV
print(f"Exporting data table to {output_csv_path}...")
out_dir = os.path.dirname(output_csv_path)
if not os.path.exists(out_dir):
	os.makedirs(out_dir)
arcpy.conversion.ExportTable(ref_pts_pop, output_csv_path)
print("Processing chain completed successfully!")


###############
##  CLEANUP  ##
###############

#	Return Spatial Analyst license
arcpy.CheckInExtension("Spatial")

Reclassifying SMOD raster...
Masking to inhabited land extent...
Class cell counts (N_h): {11: 122740903.0, 12: 6166252.0, 13: 790603.0, 21: 2189006.0, 22: 190819.0, 23: 312970.0, 30: 651930.0}
Generating equalized stratified random points...
Extracting population values to points...
Sampled counts (n_h): {11: 200, 12: 200, 13: 200, 21: 200, 22: 200, 23: 200, 30: 200}
Calculating design weights...
Adding coordinates...
Exporting data table to C:\Users\tmbar\OneDrive\Documents\ArcGIS\Projects\pecahn_ref_distribution\outputs\ref_pts.csv...
Processing chain completed successfully!


'CheckedOut'